# Medical Chatbot - RAG Pipeline with ChromaDB



## Workflow
1. Load all 3 medical PDFs
2. Split into chunks
3. Embed with `sentence-transformers/all-MiniLM-L6-v2`
4. Store in ChromaDB (persisted to disk)
5. Teammates can load the saved DB directly — no re-embedding needed

## Books used
- **Gale Encyclopedia of Medicine 2nd Ed.** (`Medical_book.pdf`)
- **Harrison's Manual of Medicine 17th Ed.**
- **Current Essentials of Medicine 4th Ed.**

## 1. Install Dependencies

In [3]:
# Run this cell once to install everything needed
!pip install -q \
    langchain \
    langchain-community \
    langchain-chroma \
    chromadb \
    sentence-transformers \
    pypdf \
    python-dotenv

print(' All dependencies installed!')

✅ All dependencies installed!


## 2. Imports

In [6]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

print(' Imports successful!')

✅ Imports successful!


## 3. Configuration

Update the PDF paths to match where your books are stored.

In [7]:
# ─────────────────────────────────────────────
# UPDATE THESE PATHS to your actual PDF locations
# ─────────────────────────────────────────────

PDF_PATHS = [
    "/content/Medical_book (1).pdf",           # Gale Encyclopedia of Medicine
    "/content/harrison__s_-_manual_of_medicine_-_17th_edition.pdf",  # Harrison's Manual of Medicine
    "/content/Current Essentials of Medicine.pdf",  # Current Essentials
]

# Where the ChromaDB vector store will be saved
CHROMA_PERSIST_DIR = "./chroma_db"

# ChromaDB collection name
COLLECTION_NAME = "medical_books"

# Embedding model
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Chunk settings — tuned for medical text
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

print(f' ChromaDB will be saved to: {os.path.abspath(CHROMA_PERSIST_DIR)}')
print(f' PDFs to process: {len(PDF_PATHS)}')

📁 ChromaDB will be saved to: /content/chroma_db
📚 PDFs to process: 3


## 4. Load PDFs

Loads all three medical books. Metadata (`source`, `page`, `book_title`) is attached to every chunk so teammates can trace where each answer came from.

In [9]:
# Friendly names for the three books
BOOK_TITLES = {
    "Medical_book.pdf": "Gale Encyclopedia of Medicine 2nd Ed.",
    "harrisons_manual_17th.pdf": "Harrison's Manual of Medicine 17th Ed.",
    "current_essentials_medicine.pdf": "Current Essentials of Medicine 4th Ed.",
}


def load_pdfs(pdf_paths: list[str]) -> list[Document]:
    """
    Load multiple PDFs using PyPDFLoader.
    Enriches metadata with book_title for easier identification.
    Skips missing files with a warning.
    """
    all_docs: list[Document] = []

    for path in pdf_paths:
        if not os.path.exists(path):
            print(f'  File not found, skipping: {path}')
            continue

        print(f' Loading: {path} ...', end=' ')
        loader = PyPDFLoader(path)
        docs = loader.load()

        # Enrich metadata
        filename = Path(path).name
        book_title = BOOK_TITLES.get(filename, filename)
        for doc in docs:
            doc.metadata['book_title'] = book_title
            doc.metadata['source'] = path

        all_docs.extend(docs)
        print(f'✅ {len(docs)} pages loaded')

    print(f'\n Total pages loaded across all books: {len(all_docs)}')
    return all_docs


all_docs = load_pdfs(PDF_PATHS)

📖 Loading: /content/Medical_book (1).pdf ... ✅ 637 pages loaded
📖 Loading: /content/harrison__s_-_manual_of_medicine_-_17th_edition.pdf ... ✅ 1263 pages loaded
📖 Loading: /content/Current Essentials of Medicine.pdf ... ✅ 607 pages loaded

 Total pages loaded across all books: 2507


In [10]:
# Quick check - inspect the first document
if all_docs:
    print('=== First document preview ===')
    print(f'Book    : {all_docs[0].metadata.get("book_title")}')
    print(f'Page    : {all_docs[0].metadata.get("page")}')
    print(f'Content : {all_docs[0].page_content[:300]}...')
else:
    print(' No documents loaded — check your PDF paths!')

=== First document preview ===
Book    : Medical_book (1).pdf
Page    : 0
Content : ...


## 5. Chunk the Documents

`RecursiveCharacterTextSplitter` splits on paragraph → sentence → word boundaries, preserving medical context better than a naive split.

In [11]:
def split_documents(documents: list[Document],
                    chunk_size: int = CHUNK_SIZE,
                    chunk_overlap: int = CHUNK_OVERLAP) -> list[Document]:
    """
    Split documents into smaller chunks for embedding.
    chunk_overlap ensures sentences at chunk boundaries are not cut mid-context.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],  # Try paragraph, then sentence
    )
    chunks = splitter.split_documents(documents)
    return chunks


text_chunks = split_documents(all_docs)

print(f' Total chunks created : {len(text_chunks)}')
print(f'   chunk_size           : {CHUNK_SIZE} chars')
print(f'   chunk_overlap        : {CHUNK_OVERLAP} chars')
print()

# Show breakdown by book
from collections import Counter
book_counts = Counter(c.metadata.get('book_title', 'Unknown') for c in text_chunks)
print('Chunks per book:')
for title, count in book_counts.items():
    print(f'  {count:6,}  {title}')

📦 Total chunks created : 15778
   chunk_size           : 500 chars
   chunk_overlap        : 50 chars

Chunks per book:
   5,961  Medical_book (1).pdf
   7,428  harrison__s_-_manual_of_medicine_-_17th_edition.pdf
   2,389  Current Essentials of Medicine.pdf


In [12]:
# Inspect a sample chunk from each book
seen_books = set()
for chunk in text_chunks:
    title = chunk.metadata.get('book_title', 'Unknown')
    if title not in seen_books:
        seen_books.add(title)
        print(f'\n--- Sample chunk from: {title} ---')
        print(chunk.page_content[:300])
        print(f'[page {chunk.metadata.get("page", "?")}]')


--- Sample chunk from: Medical_book (1).pdf ---
The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION
[page 1]

--- Sample chunk from: harrison__s_-_manual_of_medicine_-_17th_edition.pdf ---
EDITORS
 
Anthony S. Fauci, MD, ScD(HON)
Chief, Laboratory of Immunoregulation; Director, 
National Institute of Allergy and Infectious Diseases, 
National Institutes of Health, Bethesda
 
Eugene Braunwald, MD, ScD(HON)
Distinguished Hersey Professor of Medicine, 
Harvard Medical School; Chairman, T
[page 2]

--- Sample chunk from: Current Essentials of Medicine.pdf ---
a LANGE medical book
CURRENT
ESSENTIALS
of MEDICINE
Fourth Edition
Edited by
Lawrence M. Tierney, Jr., MD
Professor of Medicine
University of California, San Francisco
Associate Chief of Medical Services
Veterans Affairs Medical Center
San Francisco, California
Sanjay Saint, MD, MPH
Associate Chief 
[page 1]


## 6. Load Embedding Model

Using the same `all-MiniLM-L6-v2` model as the original notebook — 384-dimensional embeddings, fast and accurate for medical text.

In [13]:
def get_embeddings(model_name: str = EMBEDDING_MODEL) -> HuggingFaceEmbeddings:
    """Load and return the HuggingFace sentence-transformer embedding model."""
    print(f'⏳ Loading embedding model: {model_name} ...')
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={'device': 'cpu'},   # Change to 'cuda' if you have a GPU
        encode_kwargs={'normalize_embeddings': True},  # Cosine similarity ready
    )
    print(f' Embedding model loaded!')
    return embeddings


embedding_model = get_embeddings()

# Quick test
test_vec = embedding_model.embed_query('What is diabetes?')
print(f'Test vector dimensions: {len(test_vec)}')

⏳ Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 ...


/tmp/ipykernel_4579/4287340112.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!
Test vector dimensions: 384


## 7. Build & Persist the ChromaDB Vector Store
ChromaDB embeds all chunks and saves them to `./chroma_db/`.



In [14]:
def build_vector_store(
    chunks: list[Document],
    embeddings: HuggingFaceEmbeddings,
    persist_dir: str = CHROMA_PERSIST_DIR,
    collection_name: str = COLLECTION_NAME,
    batch_size: int = 500,
) -> Chroma:
    """
    Build a ChromaDB vector store from document chunks.

    Processes in batches to avoid memory issues with large document sets.
    The DB is automatically persisted to disk at `persist_dir`.
    """
    os.makedirs(persist_dir, exist_ok=True)
    total = len(chunks)
    print(f' Building ChromaDB with {total:,} chunks ...')
    print(f'   Persist directory : {os.path.abspath(persist_dir)}')
    print(f'   Batch size        : {batch_size}')
    print()

    # Process first batch — creates the collection
    first_batch = chunks[:batch_size]
    print(f'Processing batch 1/{((total - 1) // batch_size) + 1} ({len(first_batch)} chunks)...')
    vectorstore = Chroma.from_documents(
        documents=first_batch,
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=persist_dir,
    )

    # Add remaining batches
    for i in range(batch_size, total, batch_size):
        batch = chunks[i: i + batch_size]
        batch_num = (i // batch_size) + 1
        total_batches = ((total - 1) // batch_size) + 1
        print(f'Processing batch {batch_num}/{total_batches} ({len(batch)} chunks)...')
        vectorstore.add_documents(batch)

    print(f'\n✅ ChromaDB built and saved to: {os.path.abspath(persist_dir)}')
    print(f'   Total vectors stored: {vectorstore._collection.count():,}')
    return vectorstore


# Only build if chroma_db doesn't already exist
# Remove the condition below if you want to rebuild from scratch
if not os.path.exists(CHROMA_PERSIST_DIR):
    vectorstore = build_vector_store(text_chunks, embedding_model)
else:
    print(f'ℹ️  ChromaDB already exists at {CHROMA_PERSIST_DIR}')
    print('   Loading existing store. To rebuild, delete the folder and re-run this cell.')
    vectorstore = Chroma(
        persist_directory=CHROMA_PERSIST_DIR,
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_model,
    )
    print(f'   Vectors loaded: {vectorstore._collection.count():,}')

🚀 Building ChromaDB with 15,778 chunks ...
   Persist directory : /content/chroma_db
   Batch size        : 500

Processing batch 1/32 (500 chunks)...
Processing batch 2/32 (500 chunks)...
Processing batch 3/32 (500 chunks)...
Processing batch 4/32 (500 chunks)...
Processing batch 5/32 (500 chunks)...
Processing batch 6/32 (500 chunks)...
Processing batch 7/32 (500 chunks)...
Processing batch 8/32 (500 chunks)...
Processing batch 9/32 (500 chunks)...
Processing batch 10/32 (500 chunks)...
Processing batch 11/32 (500 chunks)...
Processing batch 12/32 (500 chunks)...
Processing batch 13/32 (500 chunks)...
Processing batch 14/32 (500 chunks)...
Processing batch 15/32 (500 chunks)...
Processing batch 16/32 (500 chunks)...
Processing batch 17/32 (500 chunks)...
Processing batch 18/32 (500 chunks)...
Processing batch 19/32 (500 chunks)...
Processing batch 20/32 (500 chunks)...
Processing batch 21/32 (500 chunks)...
Processing batch 22/32 (500 chunks)...
Processing batch 23/32 (500 chunks)...

## 8. Load an Existing ChromaDB

In [15]:

vectorstore = Chroma(
    persist_directory=CHROMA_PERSIST_DIR,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
)

total_vectors = vectorstore._collection.count()
print(f' ChromaDB loaded from: {os.path.abspath(CHROMA_PERSIST_DIR)}')
print(f'   Total vectors in store: {total_vectors:,}')

 ChromaDB loaded from: /content/chroma_db
   Total vectors in store: 15,778


## 9. Build the Retriever

The retriever is the bridge between the vector store and the LLM. It finds the most relevant chunks for any query.

In [16]:
# k=3 returns the 3 most relevant chunks — increase for more context
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

print('✅ Retriever ready!')
print(f'   Search type : similarity (cosine)')
print(f'   Top-k       : 3 chunks per query')

✅ Retriever ready!
   Search type : similarity (cosine)
   Top-k       : 3 chunks per query


## 10. Test the Retriever

Let's verify the retriever finds relevant chunks from the medical books.

In [17]:
def retrieve_and_display(query: str, top_k: int = 3) -> list[Document]:
    """Retrieve relevant chunks and display them neatly."""
    print(f'🔍 Query: "{query}"')
    print('=' * 60)

    docs = retriever.invoke(query)

    for i, doc in enumerate(docs, 1):
        book = doc.metadata.get('book_title', 'Unknown')
        page = doc.metadata.get('page', '?')
        print(f'\n[Result {i}] {book} — Page {page}')
        print('-' * 40)
        print(doc.page_content[:400])
        if len(doc.page_content) > 400:
            print('...')

    return docs


# Test with a few medical queries
_ = retrieve_and_display('What is diabetes and how is it treated?')

🔍 Query: "What is diabetes and how is it treated?"

[Result 1] Medical_book (1).pdf — Page 634
----------------------------------------
that leads to crippling deformities.
Diabetes mellitus —A metabolic disease caused
by a deficiency of insulin, which is essential to
process carbohydrates in the body.
Gout—A hereditary metabolic disease that is a
form of arthritis and causes inflammation of the
joints. It is more common in men.
Inflammation—The reaction of tissue to injury.
Kinesiology—The science or study of movement.
Prognosis

...

[Result 2] Current Essentials of Medicine.pdf — Page 213
----------------------------------------
• Severe insulin resistance syndromes
• Altered mental status due to other cause
■ Treatment
• Patient education is important, emphasizing dietary management,
exercise, weight loss, self-monitoring of blood glucose, hypo-
glycemia awareness, foot and eye care
• Mild cases may be controlled initially with diet, exercise, and
weight loss
• Metformin or alterna

In [18]:
_ = retrieve_and_display('Symptoms and treatment of pneumonia')

🔍 Query: "Symptoms and treatment of pneumonia"

[Result 1] harrison__s_-_manual_of_medicine_-_17th_edition.pdf — Page 785
----------------------------------------
to those in other forms of pneumonia. 
Diagnosis Diagnosis is difﬁcult. Application of clinical criteria consistently
results in overdiagnosis of V AP. Use of quantitative cultures to discriminate be-
tween colonization and true infection by determining bacterial burden may be
helpful; the more distal in the respiratory tree the diagnostic sampling, the more
speciﬁc the results. 
TABLE 139-2 EMPIR
...

[Result 2] Current Essentials of Medicine.pdf — Page 49
----------------------------------------
37
2
Pulmonary Diseases
Acute Bacterial Pneumonia
■ Essentials of Diagnosis
• Fever, chills, dyspnea, cough with purulent sputum production;
early pleuritic pain suggests pneumococcal etiology
• Tachycardia, tachypnea; bronchial breath sounds with percussive
dullness and egophony over involved lungs
• Leukocytosis; WBC < 5000 or > 2

In [19]:
_ = retrieve_and_display('What are the side effects of beta blockers?')

🔍 Query: "What are the side effects of beta blockers?"

[Result 1] Medical_book (1).pdf — Page 489
----------------------------------------
medical problems listed in this section should make sure
their physicians are aware of their conditions.
USE OF CERTAIN MEDICINES. Taking beta blockers
with certain other drugs may affect the way the drugs
work or may increase the chance of side effects.
Side effects
The most common side effects are dizziness ,
drowsiness, lightheadedness, sleep problems, unusual
tiredness or weakness, and decreas
...

[Result 2] harrison__s_-_manual_of_medicine_-_17th_edition.pdf — Page 733
----------------------------------------
and frequency to limit tolerance and side effects of headache, light-headed-
ness, tachycardia.
Beta Blockers (See Table 128-1)
All have antianginal properties; β
1-selective agents are less likely to exacer-
bate airway or peripheral vascular disease. Dosage should be titrated to rest-
ing heart rate of 50–60 beats/min. Contraindication

## 11. Utility Functions


These helper functions make it easy to integrate the retriever into the chatbot pipeline.

In [20]:
def get_context_for_query(query: str, k: int = 3) -> str:
    """
    Returns a single formatted string of retrieved context.
    Pass this directly into your LLM prompt.

    Example usage in a chatbot:
        context = get_context_for_query(user_question)
        prompt = f"Use the following context to answer:\n{context}\n\nQuestion: {user_question}"
    """
    retriever_k = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    docs = retriever_k.invoke(query)

    context_parts = []
    for i, doc in enumerate(docs, 1):
        book = doc.metadata.get('book_title', 'Medical Reference')
        page = doc.metadata.get('page', '?')
        context_parts.append(
            f"[Source {i}: {book}, Page {page}]\n{doc.page_content}"
        )

    return "\n\n".join(context_parts)


def add_custom_document(text: str, metadata: dict = None) -> None:
    """
    Add a custom document chunk to the existing ChromaDB.
    Useful for adding new medical guidelines or custom content.

    Example:
        add_custom_document(
            "COVID-19 is caused by SARS-CoV-2...",
            {"source": "WHO Guidelines 2023", "book_title": "WHO Guidelines"}
        )
    """
    doc = Document(
        page_content=text,
        metadata=metadata or {"source": "custom", "book_title": "Custom Content"}
    )
    vectorstore.add_documents([doc])
    print(f'✅ Document added. Total vectors: {vectorstore._collection.count():,}')


def get_vectorstore_stats() -> dict:
    """Return basic stats about the current vector store."""
    count = vectorstore._collection.count()
    return {
        'total_vectors': count,
        'collection_name': COLLECTION_NAME,
        'persist_dir': os.path.abspath(CHROMA_PERSIST_DIR),
        'embedding_model': EMBEDDING_MODEL,
        'embedding_dimensions': 384,
    }


# Show stats
stats = get_vectorstore_stats()
print('📊 Vector Store Stats:')
for k, v in stats.items():
    print(f'   {k:25s}: {v}')

📊 Vector Store Stats:
   total_vectors            : 15778
   collection_name          : medical_books
   persist_dir              : /content/chroma_db
   embedding_model          : sentence-transformers/all-MiniLM-L6-v2
   embedding_dimensions     : 384


In [21]:
# Demo: formatted context string ready for LLM
sample_query = "What are the symptoms of heart failure?"
context = get_context_for_query(sample_query, k=2)
print(f'Context for: "{sample_query}"\n')
print(context[:800] + '...')

Context for: "What are the symptoms of heart failure?"

[Source 1: harrison__s_-_manual_of_medicine_-_17th_edition.pdf, Page 703]
with peripheral edema and ascites.
Physical Examination Signs of right-sided heart failure: JVD, hepatomegaly,
peripheral edema, murmur of tricuspid regurgitation. Left-sided signs may also
be present.

[Source 2: Current Essentials of Medicine.pdf, Page 65]
sided heart failure (jugular venous distention, peripheral edema,
hepatomegaly, ascites) common
• Right axis deviation, right bundle-branch block, right ventricular
strain or hypertrophy by electrocardiography
• Echocardiographic evidence of elevated right ventricular (RV) systolic
pressure with or without evidence of RV dilation or dysfunction
■ Differential Diagnosis
• Pulmonary venous hypertension (mitral stenosis, left heart failure
from any etiology)...


## 12. some paths

```python

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

CHROMA_PERSIST_DIR = "./chroma_db"   # ← copy this folder from the person who ran the pipeline
COLLECTION_NAME = "medical_books"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={'normalize_embeddings': True},
)

vectorstore = Chroma(
    persist_directory=CHROMA_PERSIST_DIR,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

# Ready to use!
docs = retriever.invoke("What is hypertension?")
```



---

## Summary

| Step | Description | Output |
|---|---|---|
| 4 | Load 3 medical PDFs | `all_docs` — list of pages |
| 5 | Split into 500-char chunks | `text_chunks` — chunked documents |
| 6 | Load embedding model | `embedding_model` (384-dim) |
| 7 | Build ChromaDB | `chroma_db/` folder on disk |
| 8 | Load existing ChromaDB | `vectorstore` (for teammates) |
| 9 | Create retriever | `retriever` — ready for RAG |
| 11 | Utility functions | `get_context_for_query()`, `add_custom_document()` |

### Next steps for the chatbot
- Connect `retriever` to an LLM (GPT-4, Claude, Llama, etc.)
- Use `get_context_for_query(user_question)` to fetch context
- Build a `LangChain RetrievalQA` or `ConversationalRetrievalChain`